1. Use other models instead of openai models in OpenAI SDK
2. Structured Outputs
3. Guardrails
4. Sandbox Agents
5. MCP

In [ ]:
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput


In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### Use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [ ]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

STEP 3: Create a model object

In [ ]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [ ]:
# create 3 agents with various models

sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [ ]:
# use tool-agents

description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [ ]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [ ]:
tools = [tool1, tool2, tool3, send_email_tool]

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [ ]:
# run orcestration with LLM via tools

with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

The agent can generate a json object instead of a pure text

1. We specify a Python object (by using Pydantic framework)
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.


In [ ]:
# Pydantic json class (output json schema)

class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [ ]:
EmailReview.model_json_schema()

In [ ]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [ ]:
# create an agent with specifying its output format

checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", 
                output_type=EmailReview)


In [ ]:
# run the agent

result = await Runner.run(checker, email)

In [ ]:
review = result.final_output
review

In [ ]:
review.is_professional

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool.

Input guardrails only run for the first input to the first Agent in Runner.run().

Output guardrails only run for the final output of the last agent.

If you have guardrails on other agents, they will never be called.

https://openai.github.io/openai-agents-python/guardrails/


In [ ]:
@output_guardrail
async def email_guardrail(ctx, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [ ]:
# create the agent

cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model,
                           output_guardrails=[email_guardrail])

In [ ]:
# run the agent

result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

Check out the trace:

https://platform.openai.com/traces

### On the other hand..

Guardrails is not a good method

1. There is simpler methods as follow
2. Using simpler methods can be used in other frameworks (langGraph, crewai, ...)

In [ ]:
# create and run the agent

simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


In [ ]:
# check the output with a simple method instead of Guardrails

result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

Check out the trace:

https://platform.openai.com/traces

## Sandbox Agents

### When should you use a Sandbox?

A Sandbox is useful when an AI agent needs to work like a real software developer or data analyst on a project. Typical use cases include:

* Developing and debugging code
* Analyzing data and generating reports
* Searching and processing large collections of documents
* Running multi-step pipelines and workflows
* Building agents that can pause their work and continue it across multiple sessions

In short, a Sandbox enables AI agents to do much more than generate text. It provides a persistent working environment where they can execute code, manipulate files, run commands, and maintain state over time, making it a key building block for creating truly autonomous AI agents.


This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

"a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace (like environment dependencies)
2. Capabilities: what it can do (agent access read, write, delete files, internet)
3. SandboxRunConfig: where it runs (cpu, gpu, ram, timeout)

In [ ]:
from pathlib import Path

from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

Example: Fix bug in a code

In [ ]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [ ]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

In [ ]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [ ]:
# create an agent to run in the sandbox

agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini",
                     default_manifest=manifest,
                     capabilities=capabilities)

In [ ]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

## MCP

The agent can use all the tools within the MCP

In [ ]:
from agents.mcp import MCPServerStreamableHttp

In [ ]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

Run without MCP

In [ ]:
agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

Run with using MCP

In [ ]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

And see the traces:

https://platform.openai.com/traces
